# Лабораторная работа №1
**Дисциплина**: Вычислительная математика  
**Студент**: Жирков В.К, группа 5130904/30001  
**Дата**: 2025  

---

## Формулировка задания
Решить систему дифференциальных уравнений методами:
1. Встроенный `RKF45` (из `scipy.integrate.solve_ivp`).
2. Метод Рунге-Кутты 3-го порядка (реализация вручную).

Исходная система:
$$
\begin{cases}
\frac{dx_1}{dt} = -130x_1 + 900x_2 + e^{-10t}, \\
\frac{dx_2}{dt} = 30x_1 - 300x_2 + \ln(1 + 100t^2).
\end{cases}
$$
Начальные условия: $x_1(0) = 3$, $x_2(0) = 0$.

## Теоретический материал
### Кубический сплайн
Функция `CubicSpline` строит гладкую кусочно-полиномиальную интерполяцию с непрерывными 1-й и 2-й производными.

### Численное интегрирование
Функция `quad` вычисляет интеграл с оценкой ошибки. Не работает для разрывов (например, в точке $x = \pi/2$).

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Система ОДУ
def system(t, x):
    x1, x2 = x
    dx1dt = -130 * x1 + 900 * x2 + np.exp(-10 * t)
    dx2dt = 30 * x1 - 300 * x2 + np.log(1 + 100 * t**2)
    return [dx1dt, dx2dt]

# Начальные условия
x0 = [3, 0]
t_span = (0, 0.15)

# Решение методом RKF45
sol_rkf45 = solve_ivp(system, t_span, x0, method='RK45', t_eval=np.arange(0, 0.15, 0.0075), atol=1e-4, rtol=1e-4)

# Метод Рунге-Кутты 3-го порядка
def runge_kutta_3rd_order(system, t_span, x0, h):
    t0, tf = t_span
    t = np.arange(t0, tf, h)
    x = np.zeros((len(t), len(x0)))
    x[0] = x0
    for i in range(1, len(t)):
        tn, xn = t[i-1], x[i-1]
        k1 = np.array(system(tn, xn)) * h
        k2 = np.array(system(tn + h/2, xn + k1/2)) * h
        k3 = np.array(system(tn + 3*h/4, xn + 3*k2/4)) * h
        x[i] = xn + (2*k1 + 3*k2 + 4*k3) / 9
    return t, x

h = 0.0075
t_rk3, x_rk3 = runge_kutta_3rd_order(system, t_span, x0, h)

# Графики
plt.figure(figsize=(10, 6))
plt.plot(sol_rkf45.t, sol_rkf45.y[0], 'b-', label='RKF45 x1')
plt.plot(sol_rkf45.t, sol_rkf45.y[1], 'g-', label='RKF45 x2')
plt.plot(t_rk3, x_rk3[:, 0], 'r--', label='RK3 x1')
plt.plot(t_rk3, x_rk3[:, 1], 'm--', label='RK3 x2')
plt.xlabel('Time')
plt.ylabel('Values')
plt.legend()
plt.title('Сравнение методов RKF45 и Рунге-Кутты 3-го порядка')
plt.grid(True)
plt.show()

## Результаты
1. **Графики решений**:
   - Синий/зелёный: метод `RKF45`.
   - Красный/розовый: метод Рунге-Кутты 3-го порядка.
2. **Вывод**:
   - Оба метода дали схожие результаты, но `RKF45` точнее за счёт адаптивного шага.
   - Полином Лагранжа точнее на гладких данных, но сплайн устойчивее при большом числе точек.